# Client Python pour les APIs ArcGIS REST du SITG (Genève)

## Couche générique

In [6]:
URL = "https://vector.sitg.ge.ch/arcgis/rest/services/SCANE_INDICE_MOYENNES_3_ANS/FeatureServer/0/query"

### Nombre couche

In [7]:
from sitg_api import get_count

print(f"Total: {get_count(URL)}")
print(f"Genève: {get_count(URL, where="COMMUNE='Genève'")}")

Total: 238554
Genève: 95540


### Couche complète sans géométrie

In [9]:
import os

import polars as pl

from sitg_api import fetch_all

features = fetch_all(
    URL,
    with_geometry=False,
    max_workers=os.cpu_count() - 1,  # default 4
)
df = pl.from_dicts([f["attributes"] for f in features], infer_schema_length=None)
print(df.head(2))

  0%|          | 0/239 [00:00<?, ?page/s]

|    |   OBJECTID |        EGID | ADRESSE              |   NPA | COMMUNE   | DESTINATION            |   NBRE_PRENEUR |   ANNEE |   SRE |   INDICE |   DATE_DEBUT_PERIODE |   DATE_FIN_PERIODE |   INDICE_MOY2 |   ANNEES_CONCERNEES_MOY_2 |   INDICE_MOY3 |   ANNEES_CONCERNEES_MOY_3 | ID_CONCESSIONNAIRE   |   DATE_SAISIE | AGENT_ENERGETIQUE_1   |   QUANTITE_AGENT_ENERGETIQUE_1 | UNITE_AGENT_ENERGETIQUE_1   |   AGENT_ENERGETIQUE_2 |   QUANTITE_AGENT_ENERGETIQUE_2 |   UNITE_AGENT_ENERGETIQUE_2 |   AGENT_ENERGETIQUE_3 |   QUANTITE_AGENT_ENERGETIQUE_3 |   UNITE_AGENT_ENERGETIQUE_3 | BAT_C_ADR1_REPONDANT   | BAT_C_ADR2_REPONDANT   | BAT_C_ADR3_REPONDANT   | BAT_C_ADR4_REPONDANT   | BAT_C_ADR5_REPONDANT   |   Shape__Area |   Shape__Length |
|---:|-----------:|------------:|:---------------------|------:|:----------|:-----------------------|---------------:|--------:|------:|---------:|---------------------:|-------------------:|--------------:|--------------------------:|--------------:|----------

In [10]:
print(df.to_pandas().head(2).to_markdown())

|    |   OBJECTID |        EGID | ADRESSE              |   NPA | COMMUNE   | DESTINATION            |   NBRE_PRENEUR |   ANNEE |   SRE |   INDICE |   DATE_DEBUT_PERIODE |   DATE_FIN_PERIODE |   INDICE_MOY2 |   ANNEES_CONCERNEES_MOY_2 |   INDICE_MOY3 |   ANNEES_CONCERNEES_MOY_3 | ID_CONCESSIONNAIRE   |   DATE_SAISIE | AGENT_ENERGETIQUE_1   |   QUANTITE_AGENT_ENERGETIQUE_1 | UNITE_AGENT_ENERGETIQUE_1   |   AGENT_ENERGETIQUE_2 |   QUANTITE_AGENT_ENERGETIQUE_2 |   UNITE_AGENT_ENERGETIQUE_2 |   AGENT_ENERGETIQUE_3 |   QUANTITE_AGENT_ENERGETIQUE_3 |   UNITE_AGENT_ENERGETIQUE_3 | BAT_C_ADR1_REPONDANT   | BAT_C_ADR2_REPONDANT   | BAT_C_ADR3_REPONDANT   | BAT_C_ADR4_REPONDANT   | BAT_C_ADR5_REPONDANT   |   Shape__Area |   Shape__Length |
|---:|-----------:|------------:|:---------------------|------:|:----------|:-----------------------|---------------:|--------:|------:|---------:|---------------------:|-------------------:|--------------:|--------------------------:|--------------:|----------

### Couche complète avec géométrie

In [ ]:
import os

import geopandas as gpd
from shapely.geometry import Polygon

from sitg_api import fetch_all

features_geom = fetch_all(
    URL,
    with_geometry=True,
    max_workers=os.cpu_count() - 1,  # default 4
)
gdf = gpd.GeoDataFrame(
    [f["attributes"] for f in features_geom],
    geometry=[Polygon(f["geometry"]["rings"][0]) for f in features_geom],
    crs="EPSG:2056",
)
print(gdf.head(2))

  0%|          | 0/239 [00:00<?, ?page/s]

   OBJECTID       EGID               ADRESSE   NPA COMMUNE  \
0      1001  1000577.0  Avenue de Gennecy 40  1237  Avully   
1      1002  1000577.0  Avenue de Gennecy 40  1237  Avully   

               DESTINATION  NBRE_PRENEUR  ANNEE    SRE  INDICE  ...  \
0  Hab plusieurs logements           6.0   2022  567.0   595.0  ...   
1  Hab plusieurs logements           6.0   2023  567.0   548.0  ...   

   QUANTITE_AGENT_ENERGETIQUE_3  UNITE_AGENT_ENERGETIQUE_3  \
0                           NaN                        NaN   
1                           NaN                        NaN   

   BAT_C_ADR1_REPONDANT BAT_C_ADR2_REPONDANT  BAT_C_ADR3_REPONDANT  \
0                  None                 None                  None   
1                  None                 None                  None   

  BAT_C_ADR4_REPONDANT BAT_C_ADR5_REPONDANT  Shape__Area Shape__Length  \
0                 None                 None   212.379699     58.480777   
1                 None                 None   212.379

In [13]:
print(gdf.head(2).to_markdown())

|    |   OBJECTID |        EGID | ADRESSE              |   NPA | COMMUNE   | DESTINATION             |   NBRE_PRENEUR |   ANNEE |   SRE |   INDICE |   DATE_DEBUT_PERIODE |   DATE_FIN_PERIODE |   INDICE_MOY2 | ANNEES_CONCERNEES_MOY_2   |   INDICE_MOY3 | ANNEES_CONCERNEES_MOY_3   | ID_CONCESSIONNAIRE   |   DATE_SAISIE | AGENT_ENERGETIQUE_1   |   QUANTITE_AGENT_ENERGETIQUE_1 | UNITE_AGENT_ENERGETIQUE_1   |   AGENT_ENERGETIQUE_2 |   QUANTITE_AGENT_ENERGETIQUE_2 |   UNITE_AGENT_ENERGETIQUE_2 |   AGENT_ENERGETIQUE_3 |   QUANTITE_AGENT_ENERGETIQUE_3 |   UNITE_AGENT_ENERGETIQUE_3 | BAT_C_ADR1_REPONDANT   | BAT_C_ADR2_REPONDANT   | BAT_C_ADR3_REPONDANT   | BAT_C_ADR4_REPONDANT   | BAT_C_ADR5_REPONDANT   |   Shape__Area |   Shape__Length | geometry                                                                                                                                                                                                                                                            

### Couche filtrée

In [14]:
import polars as pl

from sitg_api import fetch_all

features = fetch_all(
    URL,
    where="COMMUNE='Avully'",
)
df = pl.from_dicts([f["attributes"] for f in features], infer_schema_length=None)

print(df.head(2))

  0%|          | 0/2 [00:00<?, ?page/s]

shape: (2, 34)
┌──────────┬────────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬───────────┐
│ OBJECTID ┆ EGID       ┆ ADRESSE    ┆ NPA  ┆ … ┆ BAT_C_ADR4 ┆ BAT_C_ADR5 ┆ Shape__Are ┆ Shape__Le │
│ ---      ┆ ---        ┆ ---        ┆ ---  ┆   ┆ _REPONDANT ┆ _REPONDANT ┆ a          ┆ ngth      │
│ i64      ┆ f64        ┆ str        ┆ str  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│          ┆            ┆            ┆      ┆   ┆ null       ┆ null       ┆ f64        ┆ f64       │
╞══════════╪════════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 1577     ┆ 1.000617e6 ┆ Route      ┆ 1237 ┆ … ┆ null       ┆ null       ┆ 57.031211  ┆ 30.953325 │
│          ┆            ┆ d'Epeisses ┆      ┆   ┆            ┆            ┆            ┆           │
│          ┆            ┆ 7          ┆      ┆   ┆            ┆            ┆            ┆           │
│ 1578     ┆ 1.000617e6 ┆ Route      ┆ 1237 ┆ … ┆ null       ┆ null       ┆ 

In [15]:
print(df.to_pandas().head(2).to_markdown())

|    |   OBJECTID |        EGID | ADRESSE            |   NPA | COMMUNE   | DESTINATION            |   NBRE_PRENEUR |   ANNEE |   SRE |   INDICE |   DATE_DEBUT_PERIODE |   DATE_FIN_PERIODE |   INDICE_MOY2 | ANNEES_CONCERNEES_MOY_2   |   INDICE_MOY3 | ANNEES_CONCERNEES_MOY_3   | ID_CONCESSIONNAIRE   |   DATE_SAISIE | AGENT_ENERGETIQUE_1   |   QUANTITE_AGENT_ENERGETIQUE_1 | UNITE_AGENT_ENERGETIQUE_1   |   AGENT_ENERGETIQUE_2 |   QUANTITE_AGENT_ENERGETIQUE_2 |   UNITE_AGENT_ENERGETIQUE_2 |   AGENT_ENERGETIQUE_3 |   QUANTITE_AGENT_ENERGETIQUE_3 |   UNITE_AGENT_ENERGETIQUE_3 | BAT_C_ADR1_REPONDANT   | BAT_C_ADR2_REPONDANT   | BAT_C_ADR3_REPONDANT   | BAT_C_ADR4_REPONDANT   | BAT_C_ADR5_REPONDANT   |   Shape__Area |   Shape__Length |
|---:|-----------:|------------:|:-------------------|------:|:----------|:-----------------------|---------------:|--------:|------:|---------:|---------------------:|-------------------:|--------------:|:--------------------------|--------------:|:-------------

## IDC

In [16]:
from sitg_api.idc import fetch_idc_data

df, failures = fetch_idc_data(egid=[1015054, 1015052])

# Erreurs de validation
if len(failures) > 0:
    display(failures.counts())

print(df.head(2))

  0%|          | 0/1 [00:00<?, ?page/s]

shape: (2, 25)
┌─────────┬───────┬────────┬──────┬───┬───────────────┬─────────────┬───────────────┬──────────────┐
│ egid    ┆ annee ┆ indice ┆ sre  ┆ … ┆ annees_concer ┆ indice_moy3 ┆ annees_concer ┆ nbre_preneur │
│ ---     ┆ ---   ┆ ---    ┆ ---  ┆   ┆ nees_moy_2    ┆ ---         ┆ nees_moy_3    ┆ ---          │
│ i64     ┆ i64   ┆ i64    ┆ i64  ┆   ┆ ---           ┆ i64         ┆ ---           ┆ i64          │
│         ┆       ┆        ┆      ┆   ┆ str           ┆             ┆ str           ┆              │
╞═════════╪═══════╪════════╪══════╪═══╪═══════════════╪═════════════╪═══════════════╪══════════════╡
│ 1015052 ┆ 2011  ┆ 591    ┆ 2409 ┆ … ┆ null          ┆ null        ┆ null          ┆ 29           │
│ 1015052 ┆ 2012  ┆ 572    ┆ 2409 ┆ … ┆ null          ┆ null        ┆ null          ┆ 29           │
└─────────┴───────┴────────┴──────┴───┴───────────────┴─────────────┴───────────────┴──────────────┘


In [17]:
print(df.to_pandas().head(2).to_markdown())

|    |    egid |   annee |   indice |   sre | adresse                  |   npa | commune   | destination          | agent_energetique_1   |   quantite_agent_energetique_1 | unite_agent_energetique_1   |   agent_energetique_2 |   quantite_agent_energetique_2 |   unite_agent_energetique_2 |   agent_energetique_3 |   quantite_agent_energetique_3 |   unite_agent_energetique_3 | date_debut_periode   | date_fin_periode    | date_saisie         |   indice_moy2 |   annees_concernees_moy_2 |   indice_moy3 |   annees_concernees_moy_3 |   nbre_preneur |
|---:|--------:|--------:|---------:|------:|:-------------------------|------:|:----------|:---------------------|:----------------------|-------------------------------:|:----------------------------|----------------------:|-------------------------------:|----------------------------:|----------------------:|-------------------------------:|----------------------------:|:---------------------|:--------------------|:--------------------|--------

In [18]:
from sitg_api.idc import fetch_idc_data_pivot_egid

df_pivot_idc_an = fetch_idc_data_pivot_egid(egid=[1015054, 1015052])

print(df_pivot_idc_an.head(2))

  0%|          | 0/1 [00:00<?, ?page/s]

shape: (2, 226)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ egid    ┆ adresse_20 ┆ adresse_2 ┆ adresse_2 ┆ … ┆ unite_age ┆ unite_age ┆ unite_age ┆ unite_age │
│ ---     ┆ 11         ┆ 012       ┆ 013       ┆   ┆ nt_energe ┆ nt_energe ┆ nt_energe ┆ nt_energe │
│ i64     ┆ ---        ┆ ---       ┆ ---       ┆   ┆ tique_3_2 ┆ tique_3_2 ┆ tique_3_2 ┆ tique_3_2 │
│         ┆ str        ┆ str       ┆ str       ┆   ┆ 022       ┆ 023       ┆ 024       ┆ 025       │
│         ┆            ┆           ┆           ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆           ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1015052 ┆ Avenue Giu ┆ Avenue    ┆ Avenue    ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│         ┆ seppe-MOTT ┆ Giuseppe- ┆ Giuseppe- ┆   ┆           ┆           

In [19]:
print(df_pivot_idc_an.to_pandas().head(2).to_markdown())

|    |    egid | adresse_2011             | adresse_2012             | adresse_2013             | adresse_2014             | adresse_2015             | adresse_2016             | adresse_2017             | adresse_2018             | adresse_2019             | adresse_2020             | adresse_2021             | adresse_2022             | adresse_2023             | adresse_2024             | adresse_2025             | agent_energetique_1_2011   | agent_energetique_1_2012   | agent_energetique_1_2013   | agent_energetique_1_2014                  | agent_energetique_1_2015                  | agent_energetique_1_2016                  | agent_energetique_1_2017                  | agent_energetique_1_2018                  | agent_energetique_1_2019                  | agent_energetique_1_2020                  | agent_energetique_1_2021                  | agent_energetique_1_2022                  | agent_energetique_1_2023                  | agent_energetique_1_2024                  | agent_e